In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

1.

In [ ]:
cumulative_dates = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(cum_buildings=lambda d: d["num_buildings"].cumsum())
    .assign(cum_share=lambda d: d["cum_buildings"].div(len(data_gdynia)))
)
cumulative_dates

In [ ]:
(cumulative_dates["cum_share"] > 0.50).idxmax()

2.

In [ ]:
date_resample = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(num_diff=lambda d: d["num_buildings"].diff())
)
date_resample

In [ ]:
data_gdynia.groupby("date_of_checking").agg(
    num_buildings=("osm_id", "size"),
).assign(num_diff=lambda d: d["num_buildings"].diff())

In [ ]:
print(
    f"the biggest decline was: {date_resample.loc[date_resample['num_diff'].idxmin(), 'num_diff']} on the day of: {date_resample['num_diff'].idxmin()}"
)
print(
    f"the biggest increase was: {date_resample.loc[date_resample['num_diff'].idxmax(), 'num_diff']} on the day of: {date_resample['num_diff'].idxmax()}"
)

In [ ]:
assert (
    date_resample["num_diff"].fillna(date_resample["num_buildings"].iloc[0]).cumsum()
    == date_resample["num_buildings"]
).all()

3.

In [ ]:
date_resample_shift = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(num_prev_day=lambda d: d["num_buildings"].shift())
    .assign(num_diff=lambda d: d["num_buildings"] - d["num_prev_day"])
    .assign(day_perc_change=lambda d: d["num_buildings"] / d["num_prev_day"] - 1)
    .assign(pct_change=lambda d: d["num_buildings"].pct_change())
)
date_resample_shift

In [ ]:
assert np.isinf(date_resample_shift["pct_change"]).sum() == 1
assert date_resample_shift["pct_change"].isna().sum() == 1

pd.testing.assert_series_equal(
    date_resample_shift["day_perc_change"],
    date_resample_shift["pct_change"],
    check_names=False,
)

pd.testing.assert_series_equal(
    date_resample_shift["num_diff"], date_resample["num_diff"]
)

4.

In [ ]:
str_date_cumsum = (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .assign(num_cumsum=lambda d: d.groupby("addr_street").cumsum())
)
str_date_cumsum.tail()

In [ ]:
assert (str_date_cumsum.loc["Świętojańska", "num_cumsum"] == [1, 11, 46, 57, 106]).all()
assert (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .cumsum()
    .loc["Świętojańska", "num_buildings"][-1:]
    == 710
).all()

assert (
    data_gdynia.groupby("addr_street")["date_of_checking"].nunique().eq(1).sum() == 16
)

pd.testing.assert_series_equal(
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .groupby("addr_street")
    .cumsum()
    .groupby("addr_street")
    .tail(1)
    .sort_values("num_buildings", ascending=False)
    .reset_index()
    .set_index("addr_street")
    .loc[:, "num_buildings"]
    .sort_index(),
    data_gdynia["addr_street"].value_counts().sort_index(),
    check_names=False,
)


5.

In [ ]:
date_visit_diff = (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .reset_index(name="num_build")
    .assign(
        visit_diff=lambda d: d.groupby("addr_street")["date_of_checking"].diff().dt.days
    )
)
date_visit_diff.head()

In [ ]:
assert len(date_visit_diff) == 96
assert date_visit_diff.isna().sum().sum() == 44
assert (date_visit_diff.groupby("addr_street").size() >= 2).sum() == 28
assert date_visit_diff["visit_diff"].value_counts().to_dict() == {
    1: 24,
    2: 15,
    3: 11,
    4: 2,
}
assert (
    date_visit_diff.loc[date_visit_diff["visit_diff"] == 4, "addr_street"]
    == ["Antoniego Hryniewickiego", "Plac Konstytucji"]
).all()
assert (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .reset_index()
    .loc[:, "date_of_checking"]
    .diff()
    .dt.days.lt(0)
    .sum()
    == 27
)

6.

In [ ]:
date_visit_diff2 = date_visit_diff.assign(
    prev_visit=lambda d: d.groupby("addr_street")["date_of_checking"].shift()
).assign(visit_diff2=lambda d: (d["date_of_checking"] - d["prev_visit"]).dt.days)

In [ ]:
pd.testing.assert_series_equal(
    date_visit_diff2["visit_diff"], date_visit_diff2["visit_diff2"], check_names=False
)

assert (
    date_visit_diff2.loc[
        date_visit_diff2["addr_street"] == "Świętojańska", "date_of_checking"
    ]
    == pd.to_datetime(
        ["2026-09-24", "2026-09-26", "2026-09-27", "2026-09-29", "2026-09-30"]
    )
).all()


In [ ]:
date_visit_diff2

In [ ]:
data_gdynia.groupby(['addr_street','arch'])